# LingkodAI demo: spoken question in, spoken answer out

A recorded question goes through all seven stages: audio language ID, speech
recognition, text language ID and routing, translation to English, the
knowledge-base answer, translation back, and speech. Every stage's output is
shown, then the spoken answer plays.

Questions come from the golden conversation (24 clips, 8 per language, the
same ones the golden check uses) or from a file you upload. The first
question of a conversation decides its language; follow-ups keep it, as in
the full pipeline.

**Before you start**

- Runtime: GPU, **L4** (24 GB). Every model stays loaded between questions;
  their combined size is not measured yet, so free GPU memory prints after
  each load. If memory runs out, set `KEEP_LOADED = False`.
- A Colab Secret named `HF_TOKEN` (key icon in the left sidebar, notebook
  access on) that can read the three `lingkodai` ASR model repos and
  `lingkodai/lingkodai-eval-assets`.
- The first question downloads and loads the models (roughly 35 GB on a
  fresh runtime) and can take 10 minutes or more.

**Voice notice.** Synthesized audio here uses a cloned voice from a Philippine
Languages Database research speaker, under a CC-BY-NC, research-only licence.
It is not a real PRC official. Keep the audio and any saved sessions private.

## 1. Setup

On Colab this clones the `demo-notebooks` branch and installs the pipeline (a few
minutes), then reads the `HF_TOKEN` Secret. On your own machine it uses your
LingkodAI checkout (`~/LingkodAI`, or set `LINGKODAI_DIR`) with its
environment already installed, and reads its `.env`.

In [ ]:
BRANCH = "demo-notebooks"
import os, re, subprocess, sys
def run(cmd, log=None):
    """Run a command and print its output here; raise if it fails. With log=,
    everything also goes to that file and compile progress lines are skipped."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd), flush=True)
    sink = open(log, "w", encoding="utf-8") if log else None
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        if sink:
            sink.write(line)
        if not (sink and re.match(r"^\[ *\d+%\]", line)):
            print(line, end="", flush=True)
    if sink:
        sink.close()
    if proc.wait() != 0:
        raise RuntimeError(f"exit code {proc.returncode}: {' '.join(cmd)}" + (f" (full log: {log})" if log else ""))
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO = "/content/LingkodAI"
    if not os.path.isdir(REPO):
        run(["git", "clone", "-q", "--branch", BRANCH, "https://github.com/isabelbeab/LingkodAI.git", REPO])
    os.chdir(REPO)
    run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[gpu,demo]"])
    from google.colab import userdata
    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception as exc:  # reported, not hidden: later steps that need it check again
        print(f"HF_TOKEN Secret not read ({type(exc).__name__}): the next step needs it")
else:
    REPO = os.path.abspath(os.path.expanduser(os.environ.get("LINGKODAI_DIR", "~/LingkodAI")))
    os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
from src import demo_ui
if not IN_COLAB:
    print(".env set:", demo_ui.load_env_file(".env") or "nothing new")

import torch, transformers
assert transformers.__version__ == "4.57.3", (
    f"transformers {transformers.__version__} is loaded, not 4.57.3: restart the runtime (or kernel) and run from step 1")
run(["git", "log", "--oneline", "-1"])
print("Colab" if IN_COLAB else "local", "|", REPO, "| torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## 2. Settings

In [ ]:
SPEAK = True          # speak each answer (needs HF_TOKEN for the reference voices)
TTS_BACKEND = "cpp"   # "cpp": qwentts.cpp, fast | "hf": qwen-tts, the evaluated path, slow
QUANT = "Q8_0"        # cpp weights: "Q8_0", or "F32" (about 5 GB more GPU memory)
KEEP_LOADED = True    # False on a smaller GPU: frees each model after use (much slower turns)

## 3. Speech recognition and the golden clips

Fetches the eval assets (the golden clips, DS6 and the reference voices, each
checked against its SHA-256), and builds patched local copies of the ASR
checkpoints: the Hub copies do not load under transformers 4.57.3
(`CHANGELOG.md`, 2026-09-22). Locally, `ASR_CEB`, `ASR_FIL` and `ASR_ENG` in
`.env` skip the patch step.

In [ ]:
import shutil

assert os.environ.get("HF_TOKEN"), "HF_TOKEN is needed for the ASR checkpoints and the eval assets"
assert shutil.which("ffmpeg"), "ffmpeg is needed to read audio files (sudo apt-get install ffmpeg)"
assert "src.asr" not in sys.modules, "src.asr is already imported: restart the runtime and run from step 1"
ASSETS = f"{REPO}/outputs/eval_assets"
run([sys.executable, "scripts/fetch_eval_assets.py", "--out", ASSETS])
DS6 = f"{ASSETS}/golden/DS6_augmented.xlsx"

LANGS = ("ceb", "fil", "eng")
if all(os.path.isdir(os.environ.get(f"ASR_{lang.upper()}", "")) for lang in LANGS):
    print("Using patched ASR checkpoints from the environment")
else:
    run([sys.executable, "scripts/patch_asr_tokenizers.py", "--out", f"{REPO}/outputs/asr_patched"])
    for lang in LANGS:
        os.environ[f"ASR_{lang.upper()}"] = f"{REPO}/outputs/asr_patched/{lang}"
for lang in LANGS:
    print(f"ASR_{lang.upper()}={os.environ[f'ASR_{lang.upper()}']}")

## 4. Voice

Skipped when `SPEAK` is False. Fetches the three reference voices from the
private eval assets. For `TTS_BACKEND = "cpp"` it also builds the patched
`tts-server`, or restores an earlier build from the cache (on your Drive on Colab, `outputs/qwentts_cache/` locally; the first
build takes several minutes), and downloads the `QUANT` weights with their
checksums checked.

In [ ]:
ASSETS = f"{REPO}/outputs/eval_assets"
TTS_OPTIONS = None
if not SPEAK:
    print("SPEAK is False: no voice setup")
else:
    assert os.environ.get("HF_TOKEN"), "Speech needs HF_TOKEN: the reference voices are in the private eval assets"
    run([sys.executable, "scripts/fetch_eval_assets.py", "--out", ASSETS])
    os.environ["TTS_REF_DIR"] = f"{ASSETS}/tts_ref"
    if TTS_BACKEND == "cpp":
        if IN_COLAB:
            from google.colab import drive
            drive.mount("/content/drive")
            cache, gguf = "/content/drive/MyDrive/lingkodai_cache", "/content/models/gguf"
        else:
            cache, gguf = f"{REPO}/outputs/qwentts_cache", f"{REPO}/outputs/gguf"
        run(["bash", "scripts/build_qwentts.sh", "--cuda-arch", "native", "--cache-dir", cache],
            log=f"{REPO}/outputs/build_qwentts.log")
        run([sys.executable, "scripts/fetch_qwentts_weights.py", "--out", gguf, "--quant", QUANT])
        TTS_OPTIONS = {"mode": "segmented", "log_dir": f"{REPO}/outputs/demo_tts",
                       "talker_gguf": f"{gguf}/qwen-talker-1.7b-base-{QUANT}.gguf",
                       "codec_gguf": f"{gguf}/qwen-tokenizer-12hz-{QUANT}.gguf"}
    print("Voice ready:", TTS_BACKEND, QUANT if TTS_BACKEND == "cpp" else "")

## 5. Start the session

Nothing loads yet: each model loads the first time a question needs it and
then stays loaded, so the first question is slow (it also downloads the
models on a fresh runtime) and later ones are not. Free GPU memory prints after
every load.

In [ ]:
import pandas as pd
from IPython.display import Markdown, display
from src.demo import DemoSession, gpu_memory

session = DemoSession(tts_backend=TTS_BACKEND, tts_options=TTS_OPTIONS, keep_loaded=KEEP_LOADED)
CLIPS = demo_ui.golden_clips(DS6, ASSETS)


def ask_clip(clip_id, lang=None, speak=None):
    """One golden clip as the next turn. lang= skips both language IDs on a
    conversation's first question."""
    clip = next((c for c in CLIPS if c["clip_id"] == clip_id), None)
    assert clip is not None, f"unknown clip {clip_id!r}; see the table below"
    display(Markdown(f"**DS6 reference question:** {clip['question']}"))
    turn = session.ask_audio(clip["path"], lang=lang, speak=SPEAK if speak is None else speak)
    turn.show(title=f"{clip_id} (turn {turn.record.turn_idx + 1})")


print("Ready. GPU", gpu_memory())
with pd.option_context("display.max_colwidth", 100):
    display(pd.DataFrame(CLIPS)[["clip_id", "question", "english"]])

## 6. Ask with a golden clip

The table above lists the 24 clips with DS6's reference transcript and
English reference, to compare with what the pipeline heard. Each clip is the
next turn of the current conversation; `session.reset()` starts a new one.

In [ ]:
session.reset()
ask_clip("ceb_01_mt_01")

In [ ]:
ask_clip("ceb_01_mt_02")

## 7. A whole golden conversation

All eight turns of one language's conversation, follow-ups included.

In [ ]:
LANG = "fil"  # "ceb", "fil" or "eng"
session.reset()
for clip in [c for c in CLIPS if c["lang"] == LANG]:
    ask_clip(clip["clip_id"])

## 8. Your own recording

Any audio file ffmpeg can read (m4a, mp3, wav, ...), one question in Cebuano,
Filipino or English. On Colab a file picker opens; locally, set `MY_AUDIO`.
Uploads are stored under `outputs/uploads/` on this runtime only.

In [ ]:
session.reset()
if IN_COLAB:
    from google.colab import files

    uploaded = files.upload()
    name = next(iter(uploaded))
    os.makedirs(f"{REPO}/outputs/uploads", exist_ok=True)
    MY_AUDIO = f"{REPO}/outputs/uploads/{name}"
    os.replace(name, MY_AUDIO)
else:
    MY_AUDIO = "/path/to/your/question.m4a"
session.ask_audio(MY_AUDIO, speak=SPEAK).show(title="Your question")

## 9. Save this conversation

Writes every turn's text and audio, including each spoken question, to
`MyDrive/lingkodai_demo_sessions/` on Colab, or `outputs/demo_sessions/`
locally, for `demo_replay.ipynb`. Keep the folder private.

In [ ]:
import time
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_ROOT = "/content/drive/MyDrive/lingkodai_demo_sessions"
else:
    SAVE_ROOT = f"{REPO}/outputs/demo_sessions"
session.save(f"{SAVE_ROOT}/audio-{time.strftime('%Y%m%d-%H%M%S')}")

## 10. Free the GPU

Unloads every model and stops the TTS server. Run step 5 again to start over.

In [ ]:
session.unload()